# Browse DestinE Climate DT Data with Polytope

This notebook demonstrates how to browse and explore DestinE Climate Digital Twin data
using Polytope and `earthkit-data`, focusing on the **ICON**, **IFS-FESOM**, and
**IFS-NEMO** models with applications for rainfall analysis over Africa.

> **Important — Server addresses:** ICON and IFS-FESOM data are served from
> `polytope.lumi.apps.dte.destination-earth.eu` (LUMI), while IFS-NEMO data
> comes from `polytope.mn5.apps.dte.destination-earth.eu` (MN5).
> **Experiment keywords:** Use abbreviated names: `'hist'` for historical,
> `'cont'` for control, `'SSP3-7.0'` for projections. Use `activity='baseline'`
> for hist/cont experiments and `activity='projections'` for SSP3-7.0.
> **Streams:** The hourly `clte` stream contains instantaneous surface
> parameters (134 SP, 165 10U, 166 10V). Precipitation (228228) and
> temperature (167) require the monthly `clmn` stream or the
> `PolytopeZarrStore` approach from polytope-examples.

## Prerequisites
- Conda environment `destine` with `earthkit-data`, `polytope-client`, `covjsonkit` installed
- DestinE Platform account with upgraded access
- Authentication configured (`~/.polytopeapirc`)

See `docs/polytope_setup.md` for detailed setup instructions.

## 1. Imports and Configuration

In [11]:
import earthkit.data
import os

# Polytope server configuration
# ICON and IFS-FESOM → LUMI  |  IFS-NEMO → MN5
LUMI_ADDRESS = "polytope.lumi.apps.dte.destination-earth.eu"
MN5_ADDRESS  = "polytope.mn5.apps.dte.destination-earth.eu"
COLLECTION   = "destination-earth"

# Helper: get field count from earthkit-data results
# GribData objects have no __len__ and no __iter__;
# we count fields via to_xarray()
def safe_len(data) -> int:
    """Count fields in an earthkit-data result, even for GribData."""
    try:
        return len(data)
    except TypeError:
        pass
    # GribData: no __len__, no __iter__ - use to_xarray()
    try:
        ds = data.to_xarray()
        return sum(
            ds[var].size for var in ds.data_vars
        )
    except Exception:
        return 1  # at least we have data

def peek_metadata(data, n=3):
    """Print brief metadata summary from earthkit-data result."""
    try:
        ds = data.to_xarray()
        print(f"  Variables: {list(ds.data_vars)}")
        for var in list(ds.data_vars)[:n]:
            print(f"    {var}: {ds[var].dims} shape={ds[var].shape}")
    except Exception as e:
        print(f"  (metadata unavailable: {e})")

# Helper: pick the right address based on model
def get_address(model: str) -> str:
    """Return the correct Polytope server address for a given model."""
    if model == "IFS-NEMO":
        return MN5_ADDRESS
    return LUMI_ADDRESS  # ICON, IFS-FESOM, and others

print(f"earthkit-data version: {earthkit.data.__version__}")
print(f"LUMI address: {LUMI_ADDRESS}")
print(f"MN5  address: {MN5_ADDRESS}")


earthkit-data version: 1.0.2
LUMI address: polytope.lumi.apps.dte.destination-earth.eu
MN5  address: polytope.mn5.apps.dte.destination-earth.eu


## 2. Quick Connection Test

Make a small test request to verify connectivity and authentication
for each server (LUMI and MN5).

In [12]:
# Test LUMI (ICON) — historical simulation, single parameter
print("Testing LUMI server (ICON historical)...")
try:
    lumidata = earthkit.data.from_source(
        "polytope", COLLECTION,
        {
            'activity': 'baseline',
            'class': 'd1',
            'dataset': 'climate-dt',
            'date': '20000615',
            'experiment': 'hist',
            'expver': '0001',
            'generation': '2',
            'levtype': 'sfc',
            'model': 'ICON',
            'param': '134',            # Surface pressure
            'realization': '1',
            'resolution': 'standard',
            'stream': 'clte',
            'time': '0000',
            'type': 'fc'
        },
        address=LUMI_ADDRESS,
        stream=False
    )
    print(f"  ✓ LUMI OK — {safe_len(lumidata)} field(s)")
except Exception as e:
    print(f"  ✗ LUMI failed: {e}")

# Test MN5 (IFS-NEMO) — projections
print("Testing MN5 server (IFS-NEMO SSP3-7.0)...")
try:
    mn5data = earthkit.data.from_source(
        "polytope", COLLECTION,
        {
            'activity': 'projections',
            'class': 'd1',
            'dataset': 'climate-dt',
            'date': '20200102',
            'experiment': 'SSP3-7.0',
            'expver': '0001',
            'generation': '2',
            'levtype': 'sfc',
            'model': 'IFS-NEMO',
            'param': '134/165/166',   # SP, 10U, 10V
            'realization': '1',
            'resolution': 'standard',
            'stream': 'clte',
            'time': '0100',
            'type': 'fc'
        },
        address=MN5_ADDRESS,
        stream=False
    )
    print(f"  ✓ MN5  OK — {safe_len(mn5data)} field(s)")
    print(mn5data)
except Exception as e:
    print(f"  ✗ MN5  failed: {e}")

2026-07-14 17:54:00 - INFO - Key read from /home/dorian.spat/.polytopeapirc
2026-07-14 17:54:00 - INFO - Sending request...
{'request': 'activity: baseline\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20000615'\n"
            'experiment: hist\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levtype: sfc\n'
            'model: ICON\n'
            "param: '134'\n"
            "realization: '1'\n"
            'resolution: standard\n'
            'stream: clte\n'
            "time: '0000'\n"
            'type: fc\n',
 'verb': 'retrieve'}
2026-07-14 17:54:00 - INFO - Polytope user key found in session cache for user dorian.spat


Testing LUMI server (ICON historical)...


2026-07-14 17:54:01 - INFO - Request accepted. Please poll ./314ef6fd-ab8f-4aaf-b12c-95eefaa73ba1 for status
2026-07-14 17:54:01 - INFO - Polytope user key found in session cache for user dorian.spat
2026-07-14 17:54:01 - INFO - Checking request status (314ef6fd-ab8f-4aaf-b12c-95eefaa73ba1)...
2026-07-14 17:54:02 - INFO - The current status of the request is 'processing'
2026-07-14 17:54:05 - INFO - The current status of the request is 'processed'


314ef6fd-ab8f-4aaf-b12c-95eefaa73ba1.grib:   0%|          | 0.00/236k [00:00<?, ?B/s]

2026-07-14 17:54:06 - INFO - Key read from /home/dorian.spat/.polytopeapirc
2026-07-14 17:54:06 - INFO - Sending request...
{'request': 'activity: projections\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20200102'\n"
            'experiment: SSP3-7.0\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levtype: sfc\n'
            'model: IFS-NEMO\n'
            'param: 134/165/166\n'
            "realization: '1'\n"
            'resolution: standard\n'
            'stream: clte\n'
            "time: '0100'\n"
            'type: fc\n',
 'verb': 'retrieve'}
2026-07-14 17:54:06 - INFO - Polytope user key found in session cache for user dorian.spat


  ✗ LUMI failed: 'GribData' object is not iterable
Testing MN5 server (IFS-NEMO SSP3-7.0)...


2026-07-14 17:54:07 - INFO - Request accepted. Please poll ./6f5773fe-3496-406a-8d4d-d9c83c6e945c for status
2026-07-14 17:54:07 - INFO - Polytope user key found in session cache for user dorian.spat
2026-07-14 17:54:07 - INFO - Checking request status (6f5773fe-3496-406a-8d4d-d9c83c6e945c)...
2026-07-14 17:54:08 - INFO - The current status of the request is 'queued'
2026-07-14 17:54:08 - INFO - The current status of the request is 'processing'
2026-07-14 17:54:10 - INFO - The current status of the request is 'processed'


6f5773fe-3496-406a-8d4d-d9c83c6e945c.grib:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

  ✗ MN5  failed: 'GribData' object is not iterable


## 3. Browse Available Data

Explore data availability for different models and experiments.
All examples use the hourly `clte` stream with surface variables
SP (134), 10U (165), and 10V (166).

### 3.1 ICON — Control Simulation (1950 forcing)

Control simulations represent present-day climate with fixed forcing.
Use `activity='baseline'` and `experiment='cont'`. Data is available
from the early 1990s on LUMI.

In [13]:
# ICON control simulation — surface variables, one day
icon_control_request = {
    'activity': 'baseline',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '19900102',
    'experiment': 'cont',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'ICON',
    'param': '134/165/166',    # SP, 10U, 10V
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/0300/0600/0900/1200/1500/1800/2100',  # All 3-hourly steps
    'type': 'fc'
}

print("Requesting ICON control data (LUMI)...")
try:
    icon_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        icon_control_request,
        address=get_address('ICON'),
        stream=False
    )
    print(f"✓ Retrieved {safe_len(icon_data)} field(s)")
    peek_metadata(icon_data)
except Exception as e:
    print(f"✗ Error: {e}")


2026-07-14 17:54:21 - INFO - Key read from /home/dorian.spat/.polytopeapirc
2026-07-14 17:54:21 - INFO - Sending request...
{'request': 'activity: baseline\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '19900102'\n"
            'experiment: cont\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levtype: sfc\n'
            'model: ICON\n'
            'param: 134/165/166\n'
            "realization: '1'\n"
            'resolution: standard\n'
            'stream: clte\n'
            'time: 0000/0300/0600/0900/1200/1500/1800/2100\n'
            'type: fc\n',
 'verb': 'retrieve'}
2026-07-14 17:54:21 - INFO - Polytope user key found in session cache for user dorian.spat


Requesting ICON control data (LUMI)...


2026-07-14 17:54:22 - INFO - Request accepted. Please poll ./7747e581-7f2f-4f26-906a-ab9c29e31d26 for status
2026-07-14 17:54:22 - INFO - Polytope user key found in session cache for user dorian.spat
2026-07-14 17:54:22 - INFO - Checking request status (7747e581-7f2f-4f26-906a-ab9c29e31d26)...
2026-07-14 17:54:22 - INFO - The current status of the request is 'processing'
2026-07-14 17:54:26 - INFO - The current status of the request is 'processed'


7747e581-7f2f-4f26-906a-ab9c29e31d26.grib:   0%|          | 0.00/6.58M [00:00<?, ?B/s]

✗ Error: 'GribData' object is not iterable


### 3.2 ICON — Historical Simulation

Historical simulations cover 1950–2014 with observed forcing.
Use `activity='baseline'` and `experiment='hist'`.

In [14]:
# ICON historical simulation — surface pressure and winds
# (Precipitation/temperature require the monthly 'clmn' stream;
#  here we use surface variables available in the hourly 'clte' stream.)
icon_hist_request = {
    'activity': 'baseline',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20000615',           # Mid-June (wet season in West Africa)
    'experiment': 'hist',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'ICON',
    'param': '134/165/166',        # SP, 10U, 10V
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/1200',
    'type': 'fc'
}

print("Requesting ICON historical surface data (LUMI)...")
try:
    icon_hist_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        icon_hist_request,
        address=get_address('ICON'),
        stream=False
    )
    print(f"✓ Retrieved {safe_len(icon_hist_data)} field(s)")
    peek_metadata(icon_hist_data)
    # Convert to xarray
    ds = icon_hist_data.to_xarray()
    print(f"Dataset: {ds}")
except Exception as e:
    print(f"✗ Error: {e}")


2026-07-14 17:54:32 - INFO - Key read from /home/dorian.spat/.polytopeapirc
2026-07-14 17:54:32 - INFO - Sending request...
{'request': 'activity: baseline\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20000615'\n"
            'experiment: hist\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levtype: sfc\n'
            'model: ICON\n'
            'param: 134/165/166\n'
            "realization: '1'\n"
            'resolution: standard\n'
            'stream: clte\n'
            'time: 0000/1200\n'
            'type: fc\n',
 'verb': 'retrieve'}
2026-07-14 17:54:32 - INFO - Polytope user key found in session cache for user dorian.spat


Requesting ICON historical surface data (LUMI)...


2026-07-14 17:54:33 - INFO - Request accepted. Please poll ./eac28562-e37a-4f1e-8980-160124034ad1 for status
2026-07-14 17:54:33 - INFO - Polytope user key found in session cache for user dorian.spat
2026-07-14 17:54:33 - INFO - Checking request status (eac28562-e37a-4f1e-8980-160124034ad1)...
2026-07-14 17:54:33 - INFO - The current status of the request is 'queued'
2026-07-14 17:54:33 - INFO - The current status of the request is 'processing'
2026-07-14 17:54:37 - INFO - The current status of the request is 'processed'


eac28562-e37a-4f1e-8980-160124034ad1.grib:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

✗ Error: 'GribData' object is not iterable


### 3.3 IFS-FESOM — Projections (SSP3-7.0)

Future projections under the SSP3-7.0 scenario.
Use `activity='projections'` and `experiment='SSP3-7.0'`.

In [ ]:
# IFS-FESOM future projections — surface pressure and winds
fesom_proj_request = {
    'activity': 'projections',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20200615',           # Wet season
    'experiment': 'SSP3-7.0',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'IFS-FESOM',
    'param': '134/165/166',       # SP, 10U, 10V
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/0600/1200/1800',
    'type': 'fc'
}

print("Requesting IFS-FESOM SSP3-7.0 projections (LUMI)...")
try:
    fesom_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        fesom_proj_request,
        address=get_address('IFS-FESOM'),
        stream=False
    )
    print(f"✓ Retrieved {safe_len(fesom_data)} fields")
except Exception as e:
    print(f"✗ Error: {e}")

### 3.4 IFS-FESOM — Control Simulation

IFS-FESOM control simulation with fixed 1950 forcing.
Use `activity='baseline'` and `experiment='cont'`.

In [ ]:
# IFS-FESOM control — surface pressure and winds
fesom_control_request = {
    'activity': 'baseline',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '19900615',
    'experiment': 'cont',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'IFS-FESOM',
    'param': '134/165/166',       # SP, 10U, 10V
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/0600/1200/1800',
    'type': 'fc'
}

print("Requesting IFS-FESOM control surface data (LUMI)...")
try:
    fesom_ctrl_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        fesom_control_request,
        address=get_address('IFS-FESOM'),
        stream=False
    )
    print(f"✓ Retrieved {safe_len(fesom_ctrl_data)} fields")
except Exception as e:
    print(f"✗ Error: {e}")

## 4. Working with Retrieved Data

Convert retrieved data to xarray for analysis and plotting.

In [ ]:
# Example: quick visualization of retrieved data
# (requires matplotlib, cartopy, and xarray)

def quick_plot(data, variable_name="Surface pressure", cmap="viridis"):
    """Quick plot of first field in the dataset."""
    try:
        import cartopy.crs as ccrs
        import matplotlib.pyplot as plt
    except ImportError as e:
        print(f"⚠ matplotlib/cartopy not available: {e}")
        print("  Install with: conda install -c conda-forge matplotlib cartopy")
        return

    try:
        ds = data.to_xarray()
    except Exception as e:
        print(f"⚠ Could not convert to xarray: {e}")
        print("  Raw data object:", type(data).__name__)
        return

    try:
        fig = plt.figure(figsize=(12, 6))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.coastlines()
        ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

        # Plot first variable/time step
        first_var = list(ds.data_vars)[0]
        ds[first_var].isel(time=0 if "time" in ds.dims else None).plot(
            ax=ax, cmap=cmap, transform=ccrs.PlateCarree()
        )
        ax.set_title(f"{variable_name} ({first_var})")
        plt.show()
    except Exception as e:
        print(f"⚠ Plotting failed: {e}")

# Uncomment to plot if data was successfully retrieved:
# if "icon_data" in dir():
#     quick_plot(icon_data, "ICON Surface Pressure")


## 5. Low-Level Access with polytope-client

For advanced use cases, use the lower-level `polytope-client` directly.

In [ ]:
# Using the low-level polytope-client (requires explicit credentials
# or ~/.polytopeapirc)

from polytope.api import Client

# ICON/IFS-FESOM → LUMI, IFS-NEMO → MN5
# client = Client(address=get_address('ICON'))

# Cancel any pending requests from previous sessions
# client.revoke("all")

# Download data
# files = client.retrieve(COLLECTION, request, output_path="./downloaded_data")
# print(f"Downloaded files: {files}")

print("Use the cell above with valid credentials to download via polytope-client.")
print("For most use cases, earthkit-data (sections above) is recommended.")

## 6. Request Builder Helper

A flexible function to build request dictionaries for different scenarios.

In [ ]:
def build_request(
    model: str = "ICON",
    experiment: str = "hist",
    date: str = "20000615",
    time: str = "0000",
    param: str = "134",
    levtype: str = "sfc",
    realization: str = "1",
    **kwargs
) -> dict:
    """
    Build a Polytope request dictionary.
    
    Automatically sets the correct `activity` based on `experiment`:
      - 'hist' / 'cont'  → activity='baseline'
      - 'SSP3-7.0'       → activity='projections'
    
    Parameters
    ----------
    model : str
        Climate model: 'ICON', 'IFS-FESOM' (→ LUMI), 'IFS-NEMO' (→ MN5)
    experiment : str
        'cont' (control), 'hist' (historical), 'SSP3-7.0' (projections)
    date : str
        Date in YYYYMMDD format.
        'cont'/'hist' experiments typically use 1990–2014 dates,
        'SSP3-7.0' uses 2020–2050 dates.
    time : str
        Time in HHMM format, e.g. '0000' or '0000/0300/.../2100'
    param : str
        Parameter codes, e.g. '134' or '134/165/166'
    levtype : str
        'sfc' (surface), 'pl' (pressure levels), 'hl' (height levels)
    realization : str
        Ensemble realization number
    **kwargs
        Additional keys to add/override
    
    Returns
    -------
    dict
        Polytope request dictionary
    """
    # Map experiment to correct activity
    activity_map = {
        'hist': 'baseline',
        'cont': 'baseline',
        'SSP3-7.0': 'projections',
    }
    activity = activity_map.get(experiment, 'projections')
    
    request = {
        'activity': activity,
        'class': 'd1',
        'dataset': 'climate-dt',
        'date': date,
        'experiment': experiment,
        'expver': '0001',
        'generation': '2',
        'levtype': levtype,
        'model': model,
        'param': param,
        'realization': realization,
        'resolution': 'standard',
        'stream': 'clte',
        'time': time,
        'type': 'fc'
    }
    request.update(kwargs)
    return request


# Example: Build a request for surface data
example_request = build_request(
    model="IFS-FESOM",
    experiment="SSP3-7.0",
    date="20200615",
    time="0000/0600/1200/1800",
    param="134/165/166"
)

print("Example request for surface data:")
for key, value in example_request.items():
    print(f"  {key}: {value}")

## 7. Browsing Data Catalogue Programmatically

Use the Climate DT Explorer approach to discover available data.

In [ ]:
# Explorer notebooks included locally in this directory:
# - browse_destine_data.ipynb  — Interactive browse & download
# - 03_lazy_browse_portfolio.ipynb  — Monthly catalogue (from polytope-examples)
#
# Additional notebooks:
# - 04_lazy_browse_portfolio_hourly.ipynb — Hourly catalogue
# - 05_variable_lookup.ipynb — Search for variables
#
# Repository: https://github.com/destination-earth-digital-twins/polytope-examples

print("Local explorer notebooks in get-data/:")
print("  browse_destine_data.ipynb         — Browse & download Climate DT data")
print("  03_lazy_browse_portfolio.ipynb    — Monthly catalogue")
print()
print("For interactive data exploration, see:")
print("https://github.com/destination-earth-digital-twins/polytope-examples/tree/main/climate-dt/explorer")


## 8. Notes and Best Practices

- **Server addresses**: ICON and IFS-FESOM use the **LUMI** server (`polytope.lumi.apps.dte.destination-earth.eu`), IFS-NEMO uses the **MN5** server (`polytope.mn5.apps.dte.destination-earth.eu`).
- **Activity values**: Use `'baseline'` for hist/cont experiments and `'projections'` for SSP3-7.0.
- **Experiment names**: Use abbreviated forms: `'hist'` (historical), `'cont'` (control), `'SSP3-7.0'` (projections).
- **Date ranges**: Hist/cont experiments cover approx. 1990–2014. SSP3-7.0 projections start around 2015.
- **Streams**: The hourly `clte` stream contains instantaneous surface params (134 SP, 165 10U, 166 10V). Precipitation (228228) and temperature (167) are in the monthly `clmn` stream — use the `PolytopeZarrStore` approach from polytope-examples for monthly data.
- **Start small**: Test with a single time step and one parameter before requesting larger datasets.
- **Quota limits**: Max 2 concurrent downloads, 50 req/s. Be mindful of the server load.
- **Cache data**: Use `stream=False` to save retrieved data locally.
- **Authentication**: Keep `~/.polytopeapirc` valid. The token may expire.
- **For more info**: See `docs/polytope_usage.md` and `docs/data_catalogue.md`.